# Workshop 1 — Source Profiling and Sample Validation

**Team 8 — Tsunami Monitoring** · Data Analysis Programming 2026-II

This notebook reads only saved samples from `workshop_1/samples/`. Historical NOAA data cover 2010–2025; the Pacific USGS catalog sample covers 2010–2011 with M ≥ 6. The geographic analysis uses `config/pacific_polygon.geojson`. Incremental comparisons use the saved snapshots of 7 and 12 November 2026.

## 1. Setup

In [103]:
import json
import math
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

SAMPLES = Path("../workshop_1/samples")
POLYGON = Path("../config/pacific_polygon.geojson")

stamps = [
    "20261107_050434",
    "20261107_140301",
    "20261112_033843",
    "20261116_013756"
]

## 2. Load Samples

### 2.1 NOAA HazEL — historical tsunami events

The saved response contains two pages. NOAA provides date components without an explicit timezone field in these JSON files. We interpret complete times as UTC for comparison with USGS, as in the existing project analysis; this assumption still needs confirmation in NOAA documentation. `date_utc` represents a calendar date, not an invented midnight event time.

In [104]:
noaa_files = [
    SAMPLES / "noaa_tsunamis_pacific_2010_2025.json",
    SAMPLES / "noaa_tsunamis_pacific_page2.json",
]

events = []
for file in noaa_files:
    data = json.loads(file.read_text(encoding="utf-8"))
    events.extend(data["items"])

noaa = pd.DataFrame(events)

# Map NOAA descriptor codes to the descriptions in the JSON file
def load_descriptor(name):
    """Map NOAA descriptor codes to the descriptions in a saved JSON file."""
    path = SAMPLES / f"noaa_descriptor_{name}.json"
    items = json.loads(path.read_text(encoding="utf-8"))["items"]

    descriptions = {}
    for item in items:
        code = int(item["id"])
        descriptions[code] = item["description"]

    return descriptions

causes = load_descriptor("causes")
validity = load_descriptor("validity")
regions = load_descriptor("regions")
warning_statuses = load_descriptor("warning_statuses")

noaa["cause"] = noaa["causeCode"].map(causes)
noaa["validity"] = noaa["eventValidity"].map(validity)
noaa["region"] = noaa["regionCode"].map(regions)
noaa["warning_status"] = noaa["warningStatusId"].map(warning_statuses)

# Time manipulation
noaa["date_utc"] = pd.to_datetime(noaa[["year", "month", "day"]], utc=True)
noaa["time_precision"] = "day"
noaa.loc[noaa["hour"].notna(), "time_precision"] = "hour"
noaa.loc[
    noaa["hour"].notna() & noaa["minute"].notna(),
    "time_precision"
] = "minute"
with_seconds = noaa[["hour", "minute", "second"]].notna().all(axis=1)
noaa.loc[with_seconds, "time_precision"] = "second"

# Only complete clock times become exact timestamps
noaa["time_utc"] = pd.Series(
    pd.NaT,
    index=noaa.index,
    dtype="datetime64[ns, UTC]"
)
clock_columns = ["year", "month", "day", "hour", "minute", "second"]
noaa.loc[with_seconds, "time_utc"] = pd.to_datetime(
    noaa.loc[with_seconds, clock_columns], utc=True
)

# Basic validation
print("NOAA events:", len(noaa))
print("Duplicate IDs:", noaa["id"].duplicated().sum())

# Time precision
precision = ["day", "hour", "minute", "second"]

print("\nAvailable time precision:")
print(
    noaa["time_precision"]
    .value_counts()
    .reindex(precision, fill_value=0)
)

print("\nExact timestamps:", noaa["time_utc"].notna().sum())
print("Available dates:", noaa["date_utc"].notna().sum())

print("Total columns:", len(noaa.columns))

# Preview the main fields
columns = [
    "id",
    "date_utc",
    "time_utc",
    "time_precision",
    "latitude",
    "longitude",
    "eqMagnitude",
    "maxWaterHeight",
    "cause",
    "validity",
    "region"
]

noaa[columns].head(10)

NOAA events: 221
Duplicate IDs: 0

Available time precision:
time_precision
day        23
hour        0
minute      9
second    189
Name: count, dtype: int64

Exact timestamps: 189
Available dates: 221
Total columns: 57


,id,date_utc,time_utc,time_precision,latitude,longitude,eqMagnitude,maxWaterHeight,cause,validity,region
0,4522,2010-01-03 00:00:00+00:00,2010-01-03 21:48:05.300+00:00,second,-8.743,157.477,6.6,0.01,Earthquake,Definite Tsunami,"New Caledonia, New Guinea, Solomon Is., Vanuatu"
1,4523,2010-01-03 00:00:00+00:00,2010-01-03 22:36:27.900+00:00,second,-8.799,157.346,7.1,7.52,Earthquake,Definite Tsunami,"New Caledonia, New Guinea, Solomon Is., Vanuatu"
2,4542,2010-01-05 00:00:00+00:00,2010-01-05 12:15:32.200+00:00,second,-9.019,157.551,6.8,0.03,Earthquake,Definite Tsunami,"New Caledonia, New Guinea, Solomon Is., Vanuatu"
3,4682,2010-02-27 00:00:00+00:00,2010-02-27 06:34:11.500+00:00,second,-36.122,-72.898,8.8,29.00,Earthquake,Definite Tsunami,West Coast of South America
4,4722,2010-03-11 00:00:00+00:00,2010-03-11 14:39:43.900+00:00,second,-34.290,-71.891,6.9,0.14,Earthquake,Definite Tsunami,West Coast of South America
5,4762,2010-04-13 00:00:00+00:00,NaT,day,-9.282,-77.645,NaN,23.00,Landslide,Probable Tsunami,West Coast of South America
6,4782,2010-02-26 00:00:00+00:00,2010-02-26 20:31:26.900+00:00,second,25.930,128.425,7.0,0.10,Earthquake,Probable Tsunami,Japan
7,4822,2010-05-29 00:00:00+00:00,2010-05-29 11:46:50.000+00:00,second,16.708,145.970,NaN,0.04,Volcano,Definite Tsunami,"E. Australia, New Zealand, South Pacific Is."
8,5222,2010-08-10 00:00:00+00:00,2010-08-10 05:23:44.900+00:00,second,-17.541,168.069,7.2,0.23,Earthquake,Definite Tsunami,"New Caledonia, New Guinea, Solomon Is., Vanuatu"
9,5343,2010-08-13 00:00:00+00:00,2010-08-13 21:19:33.000+00:00,second,12.484,141.476,6.9,0.07,Earthquake,Definite Tsunami,"E. Australia, New Zealand, South Pacific Is."


### 2.2 USGS FDSN catalog — earthquakes (historical)

In [105]:

usgs_file = SAMPLES / "usgs_earthquakes_2010_2011_m6_pacific.csv"

usgs_hist = pd.read_csv(usgs_file)
usgs_hist["time_utc"] = pd.to_datetime(usgs_hist["time"], errors="coerce", utc=True)
usgs_hist["updated_utc"] = pd.to_datetime(usgs_hist["updated"], errors="coerce", utc=True)
print(
    "Rows:", len(usgs_hist),
    "| period:",
    usgs_hist["time_utc"].min(),
    "to",
    usgs_hist["time_utc"].max()
)
print("Total columns:", len(usgs_hist.columns))
usgs_hist.head(10)

Rows: 343 | period: 2010-01-02 08:45:33.020000+00:00 to 2011-12-26 04:48:08.870000+00:00
Total columns: 24


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,id,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,time_utc,updated_utc
0,2011-12-26T04:48:08.870Z,-16.198,-173.862,90.0,6.0,mww,536,35.4,NaN,0.95,us,usp000jcna,2022-04-08T20:25:06.544Z,"27 km SSW of Hihifo, Tonga",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-26 04:48:08.870+00:00,2022-04-08 20:25:06.544+00:00
1,2011-12-15T10:10:07.210Z,-32.718,-179.099,32.0,6.0,mww,423,21.6,NaN,1.09,us,usp000jc7p,2022-04-08T20:24:26.506Z,south of the Kermadec Islands,earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-15 10:10:07.210+00:00,2022-04-08 20:24:26.506+00:00
2,2011-12-14T05:04:58.630Z,-7.551,146.809,135.0,7.1,mww,485,10.6,NaN,0.78,us,usp000jc5z,2022-04-08T20:24:25.265Z,"25 km SSE of Wau, Papua New Guinea",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-14 05:04:58.630+00:00,2022-04-08 20:24:25.265+00:00
3,2011-12-13T07:52:11.930Z,0.041,123.030,161.0,6.0,mww,536,13.9,NaN,0.95,us,usp000jc4z,2022-04-08T20:24:22.823Z,"55 km S of Gorontalo, Indonesia",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-13 07:52:11.930+00:00,2022-04-08 20:24:22.823+00:00
4,2011-12-11T01:47:25.560Z,17.986,-99.789,59.0,6.5,mww,578,38.6,NaN,0.93,us,usp000jc1y,2024-02-23T00:54:36.837Z,"6 km SSW of Nuevo Balsas, Mexico",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-11 01:47:25.560+00:00,2024-02-23 00:54:36.837+00:00
5,2011-12-07T22:23:09.730Z,-27.899,-70.918,20.0,6.1,mww,473,35.2,NaN,0.82,us,usp000jby0,2022-08-09T04:01:46.999Z,"76 km NNW of Vallenar, Chile",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-12-07 22:23:09.730+00:00,2022-08-09 04:01:46.999+00:00
6,2011-11-30T00:27:06.990Z,15.455,118.999,9.0,6.0,mww,370,11.5,NaN,1.01,us,usp000jbmj,2022-04-08T22:21:41.633Z,"97 km W of Palauig, Philippines",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-11-30 00:27:06.990+00:00,2022-04-08 22:21:41.633+00:00
7,2011-11-28T12:26:45.450Z,-5.480,153.733,25.0,6.1,mww,265,15.1,NaN,1.12,us,usp000jbhn,2022-04-08T22:12:43.312Z,"205 km SE of Kokopo, Papua New Guinea",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-11-28 12:26:45.450+00:00,2022-04-08 22:12:43.312+00:00
8,2011-11-24T10:25:34.030Z,41.898,142.639,38.0,6.2,mww,580,18.3,NaN,0.79,us,usp000jbb8,2024-06-17T18:31:21.717Z,"53 km SSE of Shizunai-furukawach?, Japan",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-11-24 10:25:34.030+00:00,2024-06-17 18:31:21.717+00:00
9,2011-11-23T19:24:31.470Z,37.365,141.368,34.0,6.1,mww,496,23.4,NaN,0.73,us,usp000jbaf,2022-04-08T22:22:27.625Z,"35 km ESE of Namie, Japan",earthquake,NaN,NaN,NaN,NaN,reviewed,us,us,2011-11-23 19:24:31.470+00:00,2022-04-08 22:22:27.625+00:00


### 2.3 USGS real-time feed — earthquakes (incremental)

In [108]:
# Load all USGS Real-Time snapshots
snapshots = []

for stamp in stamps:
    rt_file = SAMPLES / f"usgs_realtime_{stamp}.geojson"
    feed = json.loads(rt_file.read_text(encoding="utf-8"))

    # Convert GeoJSON to DataFrame
    df = pd.json_normalize(feed["features"])
    df.columns = df.columns.str.replace("properties.", "", regex=False)

    # Extract coordinates
    df[["lon", "lat", "depth"]] = pd.DataFrame(
        df["geometry.coordinates"].tolist(),
        index=df.index
    )

    df = df.drop(columns=["geometry.coordinates", "geometry.type", "type"])

    # Convert timestamps
    df["time_utc"] = pd.to_datetime(df["time"], unit="ms", utc=True)
    df["updated_utc"] = pd.to_datetime(df["updated"], unit="ms", utc=True)

    # Identify the snapshot
    df["snapshot"] = stamp

    snapshots.append(df)

    print(f"{rt_file.name}: {len(df)} earthquakes")

# Combine all snapshots
usgs_rt_snapshots = pd.concat(snapshots, ignore_index=True)

# Select the latest snapshot for profiling
usgs_rt = usgs_rt_snapshots[
    usgs_rt_snapshots["snapshot"] == stamps[-1]
].copy().reset_index(drop=True)

print("\nAll snapshot records:", len(usgs_rt_snapshots))
print("Reference snapshot records:", len(usgs_rt))
print("Total columns:", len(usgs_rt.columns))

usgs_rt.head(10)

usgs_realtime_20261107_050434.geojson: 714 earthquakes
usgs_realtime_20261107_140301.geojson: 726 earthquakes
usgs_realtime_20261112_033843.geojson: 649 earthquakes
usgs_realtime_20261116_013756.geojson: 679 earthquakes

All snapshot records: 2768
Reference snapshot records: 679
Total columns: 32


,id,mag,place,time,updated,tz,url,detail,felt,cdi,mmi,alert,status,tsunami,sig,net,code,ids,sources,types,nst,dmin,rms,gap,magType,title,lon,lat,depth,time_utc,updated_utc,snapshot
0,us7000tygk,4.8,"141 km ESE of Petropavlovsk-Kamchatsky, Russia",1794782299154,1794783773040,None,https://earthquake.usgs.gov/earthquakes/eventpage/us7000tygk,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/us7000tygk.geojson,NaN,NaN,NaN,NaN,reviewed,0,354,us,7000tygk,",us7000tygk,",",us,",",origin,phase-data,",29,1.805,0.70,138,mb,"M 4.8 - 141 km ESE of Petropavlovsk-Kamchatsky, Russia",160.4940,52.4937,10.000,2026-11-15 22:38:19.154+00:00,2026-11-15 23:02:53.040+00:00,20261116_013756
1,us7000tyga,4.9,"44 km S of Jurm, Afghanistan",1794778641082,1794780978040,None,https://earthquake.usgs.gov/earthquakes/eventpage/us7000tyga,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/us7000tyga.geojson,NaN,NaN,NaN,NaN,reviewed,0,369,us,7000tyga,",us7000tyga,",",us,",",origin,phase-data,",67,2.431,0.78,56,mww,"M 4.9 - 44 km S of Jurm, Afghanistan",70.9205,36.4657,191.363,2026-11-15 21:37:21.082+00:00,2026-11-15 22:16:18.040+00:00,20261116_013756
2,us7000tyg6,5.0,"256 km SSE of Ambon, Indonesia",1794775916298,1794776943040,None,https://earthquake.usgs.gov/earthquakes/eventpage/us7000tyg6,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/us7000tyg6.geojson,NaN,NaN,NaN,NaN,reviewed,0,385,us,7000tyg6,",us7000tyg6,",",us,",",origin,phase-data,",90,7.246,0.75,41,mb,"M 5.0 - 256 km SSE of Ambon, Indonesia",128.7523,-5.9449,305.989,2026-11-15 20:51:56.298+00:00,2026-11-15 21:09:03.040+00:00,20261116_013756
3,us7000tyfh,5.0,"74 km N of Pārūn, Afghanistan",1794764916959,1794766098040,None,https://earthquake.usgs.gov/earthquakes/eventpage/us7000tyfh,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/us7000tyfh.geojson,NaN,NaN,NaN,NaN,reviewed,0,385,us,7000tyfh,",us7000tyfh,",",us,",",origin,phase-data,",96,2.165,0.67,55,mww,"M 5.0 - 74 km N of Pārūn, Afghanistan",70.9503,36.0947,100.728,2026-11-15 17:48:36.959+00:00,2026-11-15 18:08:18.040+00:00,20261116_013756
4,us7000tyf4,5.1,"169 km NW of Gorontalo, Indonesia",1794757274978,1794758352040,None,https://earthquake.usgs.gov/earthquakes/eventpage/us7000tyf4,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/us7000tyf4.geojson,NaN,NaN,NaN,NaN,reviewed,0,400,us,7000tyf4,",us7000tyf4,",",us,",",origin,phase-data,",97,6.299,0.88,65,mb,"M 5.1 - 169 km NW of Gorontalo, Indonesia",122.1345,1.7527,487.384,2026-11-15 15:41:14.978+00:00,2026-11-15 15:59:12.040+00:00,20261116_013756
5,us7000tyes,4.6,"27 km ESE of Union, Philippines",1794752272535,1794771479040,None,https://earthquake.usgs.gov/earthquakes/eventpage/us7000tyes,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/us7000tyes.geojson,NaN,NaN,NaN,NaN,reviewed,0,326,us,7000tyes,",us7000tyes,",",us,",",origin,phase-data,",41,2.715,0.74,114,mb,"M 4.6 - 27 km ESE of Union, Philippines",126.3512,9.6916,10.000,2026-11-15 14:17:52.535+00:00,2026-11-15 19:37:59.040+00:00,20261116_013756
6,us7000tyek,5.5,"9 km E of Union, Philippines",1794750095701,1794784799141,None,https://earthquake.usgs.gov/earthquakes/eventpage/us7000tyek,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/us7000tyek.geojson,14.0,4.6,6.399,green,reviewed,0,472,us,7000tyek,",us7000tyek,",",us,",",dyfi,ground-failure,losspager,origin,phase-data,shakemap,",71,2.742,0.82,58,mb,"M 5.5 - 9 km E of Union, Philippines",126.2000,9.7595,9.971,2026-11-15 13:41:35.701+00:00,2026-11-15 23:19:59.141+00:00,20261116_013756
7,us7000tyd3,4.7,"92 km NW of Houma, Tonga",1794728197211,1794729489040,None,https://earthquake.usgs.gov/earthquakes/eventpage/us7000tyd3,https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/us7000tyd3.geojson,NaN,NaN,NaN,NaN,reviewed,0,340,us,7000tyd3,",us7000tyd3,",",us,",",origin,phase-data,",48,5.852,0.63,46,mb,"M 4.7 - 92 km NW of Houma, Tonga",-175.9354,-20.5890,177.524,2026-11-15 07:36:37.211+00:00,2026-11-15 07:58:09.040+00:00,20261116_013756
8,us7000

### 2.4 tsunami.gov — Atom feeds and CAP messages (incremental)

CAP `msgType=Alert` is a message type; it does not by itself mean an active tsunami warning or an observed tsunami.

In [100]:
NS = {
    "a": "http://www.w3.org/2005/Atom",
    "geo": "http://www.w3.org/2003/01/geo/wgs84_pos#",
    "cap": "urn:oasis:names:tc:emergency:cap:1.2",
}

def parse_cap(path):
    """Read a CAP XML file and return its fields as a dictionary."""
    root = ET.parse(path).getroot()
    info = root.find("cap:info", NS)
    row = {}

    # General message fields
    general_fields = [
        "identifier", "sender", "sent", "status",
        "msgType", "source", "scope", "code", "incidents"
    ]

    for tag in general_fields:
        row[tag] = root.findtext(f"cap:{tag}", namespaces=NS)

    # Alert information
    if info is not None:
        info_fields = [
            "category", "event", "responseType",
            "urgency", "severity", "certainty",
            "expires", "senderName", "headline",
            "description", "instruction", "web"
        ]

        for tag in info_fields:
            row[tag] = info.findtext(f"cap:{tag}", namespaces=NS)

        # Additional parameters
        for param in info.findall("cap:parameter", NS):
            name = param.findtext("cap:valueName", namespaces=NS)
            value = param.findtext("cap:value", namespaces=NS)

            if name is not None:
                row[name] = value

        # Geographic information
        area = info.find("cap:area", NS)

        if area is not None:
            row["areaDesc"] = area.findtext("cap:areaDesc", namespaces=NS)
            row["circle"] = area.findtext("cap:circle", namespaces=NS)

        # Associated resources
        resource = info.find("cap:resource", NS)

        if resource is not None:
            row["resourceDesc"] = resource.findtext("cap:resourceDesc", namespaces=NS)
            row["mimeType"] = resource.findtext("cap:mimeType", namespaces=NS)
            row["resourceUri"] = resource.findtext("cap:uri", namespaces=NS)

    return row

def read_center(center, stamp):
    """Read an Atom snapshot and its associated CAP messages."""

    atom_file = SAMPLES / f"{center}_{stamp}.xml"
    root = ET.parse(atom_file).getroot()

    rows = []
    for number, entry in enumerate(root.findall("a:entry", NS), start=1):

        # Read Atom entry
        row = {
            "center": center,
            "snapshot": stamp,
            "entry_id": entry.findtext("a:id", namespaces=NS),
            "title": entry.findtext("a:title", namespaces=NS),
            "updated": entry.findtext("a:updated", namespaces=NS),
            "lat": entry.findtext("geo:lat", namespaces=NS),
            "lon": entry.findtext("geo:long", namespaces=NS),
        }

        # Read the associated CAP message
        cap_file = SAMPLES / f"{center}_cap{number}_{stamp}.xml"

        if cap_file.exists():
            cap_data = parse_cap(cap_file)

            for key, value in cap_data.items():
                row[f"cap_{key}"] = value

        rows.append(row)

    print(f"{center}: {atom_file.name} -> {len(rows)} entries")

    return rows

stamps = [
    "20261107_050434",
    "20261107_140301",
    "20261112_033843",
]

rows = []
for stamp in stamps:
    for center in ["ntwc", "ptwc"]:
        center_rows = read_center(center, stamp)
        rows.extend(center_rows)

# Create a DataFrame containing all snapshots
tsunami_gov_snapshots = pd.DataFrame(rows)

# Convert coordinates
tsunami_gov_snapshots["lat"] = pd.to_numeric(
    tsunami_gov_snapshots["lat"], errors="coerce"
)
tsunami_gov_snapshots["lon"] = pd.to_numeric(
    tsunami_gov_snapshots["lon"], errors="coerce"
)

# Convert dates
tsunami_gov_snapshots["sent_utc"] = pd.to_datetime(
    tsunami_gov_snapshots["cap_sent"],
    errors="coerce",
    utc=True
)
tsunami_gov_snapshots["origin_utc"] = pd.to_datetime(
    tsunami_gov_snapshots["cap_EventOriginTime"],
    errors="coerce",
    utc=True
)

# Select the latest snapshot for profiling
tsunami_gov = tsunami_gov_snapshots[
    tsunami_gov_snapshots["snapshot"] == stamps[-1]
].copy().reset_index(drop=True)

# Separate warning centers for profiling
ntwc = tsunami_gov[
    tsunami_gov["center"] == "ntwc"
].copy()

ptwc = tsunami_gov[
    tsunami_gov["center"] == "ptwc"
].copy()

print("All snapshot records:", len(tsunami_gov_snapshots))
print("Reference snapshot records:", len(tsunami_gov))
print("Available columns:", len(tsunami_gov.columns))

display(tsunami_gov.T)

ntwc: ntwc_20261107_050434.xml -> 1 entries
ptwc: ptwc_20261107_050434.xml -> 1 entries
ntwc: ntwc_20261107_140301.xml -> 1 entries
ptwc: ptwc_20261107_140301.xml -> 1 entries
ntwc: ntwc_20261112_033843.xml -> 1 entries
ptwc: ptwc_20261112_033843.xml -> 1 entries
All snapshot records: 6
Reference snapshot records: 2
Available columns: 46


,0,1
center,ntwc,ptwc
snapshot,20261112_033843,20261112_033843
entry_id,urn:uuid:4825643d-1e0e-433b-aa63-03636a4cbc35,urn:uuid:96d4cecd-e727-4c5c-92d1-096abbd0d07e
title,"65 miles SW of Port Alice, British Columbia",ABOUT 72 MILES NORTHWEST OF PUERTO RICO
updated,2026-11-11T01:51:32Z,2026-11-08T06:34:57Z
lat,49.293,19.344
lon,-127.676,-67.794
cap_identifier,PAAQ-1-tobe7v,PHEB-1-26312050
cap_sender,ntwc@noaa.gov,ntwc@noaa.gov
cap_sent,2026-11-11T01:51:32-00:00,2026-11-08T06:34:57-00:00


## 3. Source Profiling

For each source, inspect row and column counts, data types, missing values, duplicate IDs, descriptive statistics, and date and coordinate ranges. The historical samples exceed 200 records; tsunami.gov provides only the latest message from each center.

In [57]:
def profile(df, name, id_column):
    """Show the same basic checks for each source table."""
    print(f"{name}: {len(df)} rows × {len(df.columns)} columns")
    print("Duplicate IDs:", df[id_column].duplicated().sum())
    print("200-record rule:", "met" if len(df) >= 200 else "complete available feed")
    df.info()

    missing = pd.DataFrame({
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(1),
    })
    print("Missing values by field:")
    display(missing.sort_values("missing_%", ascending=False))

    print("Numeric summary (including minimum and maximum):")
    display(df.select_dtypes("number").describe().T)


### 3.1 NOAA HazEL events

In [58]:
profile(noaa, "NOAA HazEL tsunami events", "id")
print("Cause distribution:")
print(noaa["cause"].value_counts(dropna=False).to_string())
print("Validity levels:")
print(noaa["validity"].value_counts(dropna=False).to_string())
print("Regions:")
print(noaa["region"].value_counts(dropna=False).to_string())
print("Historical warning statuses:")
print(noaa["warning_status"].value_counts(dropna=False).to_string())

NOAA HazEL tsunami events: 221 rows × 57 columns
Duplicate IDs: 0
200-record rule: met
<class 'pandas.DataFrame'>
RangeIndex: 221 entries, 0 to 220
Data columns (total 57 columns):
 #   Column                           Non-Null Count  Dtype              
---  ------                           --------------  -----              
 0   id                               221 non-null    int64              
 1   year                             221 non-null    int64              
 2   month                            221 non-null    int64              
 3   day                              221 non-null    int64              
 4   hour                             198 non-null    float64            
 5   minute                           198 non-null    float64            
 6   second                           189 non-null    float64            
 7   eventValidity                    221 non-null    int64              
 8   causeCode                        221 non-null    int64              
 9   

,missing,missing_%
tsMtIi,220,99.5
missingAmountOrder,220,99.5
missing,220,99.5
damageMillionsDollars,217,98.2
housesDamaged,216,97.7
missingAmountOrderTotal,215,97.3
missingTotal,215,97.3
housesDamagedAmountOrder,214,96.8
housesDestroyed,213,96.4
housesDestroyedAmountOrder,211,95.5


Numeric summary (including minimum and maximum):


,count,mean,std,min,25%,50%,75%,max
id,221.0,5707.339367,286.839499,4522.000,5577.00000,5724.0000,5914.00000,6185.000
year,221.0,2017.402715,4.854781,2010.000,2013.00000,2017.0000,2022.00000,2025.000
month,221.0,6.606335,3.573101,1.000,3.00000,7.0000,10.00000,12.000
day,221.0,15.316742,8.484700,1.000,9.00000,16.0000,22.00000,31.000
hour,198.0,11.146465,7.229646,0.000,4.00000,12.0000,17.75000,23.000
minute,198.0,31.782828,17.513398,0.000,17.25000,34.0000,47.00000,59.000
second,189.0,29.447619,17.392893,0.000,13.70000,30.8000,44.80000,59.300
eventValidity,221.0,3.755656,0.590860,-1.000,4.00000,4.0000,4.00000,4.000
causeCode,221.0,2.104072,2.515955,0.000,1.00000,1.0000,1.00000,9.000
eqMagnitude,186.0,7.016129,0.689032,4.100,6.70000,7.0000,7.40000,9.100


Cause distribution:
cause
Earthquake                  179
Landslide                    17
Meteorological                9
Volcano                       6
Earthquake and Landslide      5
Unknown                       2
Volcano and Landslide         2
Volcano and Earthquake        1
Validity levels:
validity
Definite Tsunami        179
Probable Tsunami         33
Questionable Tsunami      8
Erroneous entry           1
Regions:
region
New Caledonia, New Guinea, Solomon Is., Vanuatu      56
Japan                                                35
E. Australia, New Zealand, South Pacific Is.         32
West Coast of South America                          23
West Coast of North and Central America              22
Alaska (including Aleutian Islands)                  18
China, North and South Korea, Philippines, Taiwan    14
Kamchatka and Kuril Islands                          14
E. Indonesia and Malaysia                             6
Hawaii, Johnston Atoll, Midway I                      1
Hist

**Interpretation.** Descriptor labels make the NOAA code distributions readable. Earthquakes dominate this sample, but landslides, volcanic events, and meteorological causes also appear. The bar view uses only saved records.

In [59]:
cause_counts = noaa["cause"].value_counts().to_frame("events")
display(cause_counts.style.bar(subset=["events"], color="#4F81BD"))

,events
cause,
Earthquake,179
Landslide,17
Meteorological,9
Volcano,6
Earthquake and Landslide,5
Unknown,2
Volcano and Landslide,2
Volcano and Earthquake,1


### 3.2 USGS FDSN catalog

In [60]:
profile(usgs_hist, 'USGS FDSN catalog', 'id')

USGS FDSN catalog: 343 rows × 24 columns
Duplicate IDs: 0
200-record rule: met
<class 'pandas.DataFrame'>
RangeIndex: 343 entries, 0 to 342
Data columns (total 24 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   time             343 non-null    str                
 1   latitude         343 non-null    float64            
 2   longitude        343 non-null    float64            
 3   depth            343 non-null    float64            
 4   mag              343 non-null    float64            
 5   magType          343 non-null    str                
 6   nst              343 non-null    int64              
 7   gap              343 non-null    float64            
 8   dmin             2 non-null      float64            
 9   rms              336 non-null    float64            
 10  net              343 non-null    str                
 11  id               343 non-null    str                
 12  updated   

,missing,missing_%
magError,343,100.0
dmin,341,99.4
horizontalError,341,99.4
magNst,298,86.9
depthError,277,80.8
rms,7,2.0
time,0,0.0
latitude,0,0.0
gap,0,0.0
nst,0,0.0


Numeric summary (including minimum and maximum):


,count,mean,std,min,25%,50%,75%,max
latitude,343.0,4.150948,29.668864,-57.6640,-18.4810,-3.4870,37.1290,60.4780
longitude,343.0,55.486135,127.091964,-179.8020,-72.8880,138.5820,144.0175,179.8740
depth,343.0,77.763627,145.035512,2.8000,15.7500,27.0000,42.0000,644.6000
mag,343.0,6.376968,0.436654,6.0000,6.1000,6.2000,6.5000,9.1000
nst,343.0,380.647230,146.545906,10.0000,276.5000,384.0000,477.0000,832.0000
gap,343.0,30.757726,25.274444,9.5000,16.5000,23.0000,36.4500,239.0000
dmin,2.0,0.418100,0.135199,0.3225,0.3703,0.4181,0.4659,0.5137
rms,336.0,0.995476,0.190657,0.1100,0.8700,0.9800,1.1100,1.5000
horizontalError,2.0,1.180000,0.890955,0.5500,0.8650,1.1800,1.4950,1.8100
depthError,66.0,7.470000,4.931779,1.3100,4.3750,6.3000,9.1750,31.6100


**Magnitude distribution.** Bin historical USGS magnitudes to see the shape of the M ≥ 6 sample. Large earthquakes remain valid observations.

In [61]:
magnitude_ranges = pd.cut(usgs_hist["mag"], bins=[6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5], right=False)
magnitude_counts = magnitude_ranges.value_counts().sort_index().to_frame("earthquakes")
display(magnitude_counts.style.bar(subset=["earthquakes"], color="#6D9F71"))

,earthquakes
mag,
"[6.0, 6.5)",240
"[6.5, 7.0)",64
"[7.0, 7.5)",31
"[7.5, 8.0)",6
"[8.0, 8.5)",0
"[8.5, 9.0)",1
"[9.0, 9.5)",1


### 3.3 USGS real-time feed

In [62]:
profile(usgs_rt, 'USGS real-time feed', 'id')

USGS real-time feed: 726 rows × 31 columns
Duplicate IDs: 0
200-record rule: met
<class 'pandas.DataFrame'>
RangeIndex: 726 entries, 0 to 725
Data columns (total 31 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   id           726 non-null    str                
 1   mag          726 non-null    float64            
 2   place        726 non-null    str                
 3   time         726 non-null    int64              
 4   updated      726 non-null    int64              
 5   tz           0 non-null      object             
 6   url          726 non-null    str                
 7   detail       726 non-null    str                
 8   felt         128 non-null    float64            
 9   cdi          128 non-null    float64            
 10  mmi          85 non-null     float64            
 11  alert        74 non-null     str                
 12  status       726 non-null    str                
 13  tsunami   

,missing,missing_%
tz,726,100.0
alert,652,89.8
mmi,641,88.3
felt,598,82.4
cdi,598,82.4
mag,0,0.0
id,0,0.0
url,0,0.0
updated,0,0.0
time,0,0.0


Numeric summary (including minimum and maximum):


,count,mean,std,min,25%,50%,75%,max
mag,726.0,4.857778e+00,4.125941e-01,4.500000e+00,4.600000e+00,4.700000e+00,5.000000e+00,7.700000e+00
time,726.0,1.792529e+12,7.098019e+08,1.791469e+12,1.791837e+12,1.792577e+12,1.793053e+12,1.794051e+12
updated,726.0,1.793376e+12,4.976346e+08,1.792143e+12,1.793009e+12,1.793425e+12,1.793829e+12,1.794060e+12
felt,128.0,2.756250e+01,1.808077e+02,0.000000e+00,1.000000e+00,2.000000e+00,7.000000e+00,2.018000e+03
cdi,128.0,3.082813e+00,1.479205e+00,0.000000e+00,2.000000e+00,3.100000e+00,3.800000e+00,7.900000e+00
mmi,85.0,3.446706e+00,2.557439e+00,0.000000e+00,1.000000e+00,3.348000e+00,5.592000e+00,8.750000e+00
tsunami,726.0,8.264463e-03,9.059507e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
sig,726.0,3.689242e+02,8.848828e+01,3.120000e+02,3.260000e+02,3.400000e+02,3.850000e+02,1.790000e+03
nst,726.0,6.061983e+01,3.951627e+01,1.400000e+01,3.300000e+01,5.100000e+01,7.675000e+01,3.630000e+02
dmin,726.0,4.872938e+00,4.794292e+00,6.219000e-02,1.828250e+00,3.669000e+00,6.658000e+00,3.946700e+01


### 3.4 tsunami.gov

In [63]:
print("Entries per center:")
print(tsunami_gov.groupby("center").size())
profile(tsunami_gov, "tsunami.gov (NTWC + PTWC)", "entry_id")
print("Message type and CAP severity:")
print(tsunami_gov[["center", "cap_msgType", "cap_event", "cap_severity", "cap_certainty"]].to_string(index=False))

Entries per center:
center
ntwc    1
ptwc    1
dtype: int64
tsunami.gov (NTWC + PTWC): 2 rows × 44 columns
Duplicate IDs: 0
200-record rule: complete available feed
<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 44 columns):
 #   Column                             Non-Null Count  Dtype              
---  ------                             --------------  -----              
 0   center                             2 non-null      str                
 1   snapshot                           2 non-null      str                
 2   entry_id                           2 non-null      str                
 3   title                              2 non-null      str                
 4   updated                            2 non-null      str                
 5   lat                                2 non-null      float64            
 6   lon                                2 non-null      float64            
 7   cap_identifier                     2 non-null      str  

,missing,missing_%
cap_NWSUGC,1,50.0
cap_WEAK53,1,50.0
cap_ProductDefinition,1,50.0
center,0,0.0
updated,0,0.0
lat,0,0.0
lon,0,0.0
cap_identifier,0,0.0
cap_sender,0,0.0
snapshot,0,0.0


Numeric summary (including minimum and maximum):


,count,mean,std,min,25%,50%,75%,max
lat,2.0,34.198,8.765296,28.000,31.099,34.198,37.297,40.396
lon,2.0,7.208,187.654826,-125.484,-59.138,7.208,73.554,139.900


Message type and CAP severity:
center cap_msgType           cap_event cap_severity cap_certainty
  ntwc       Alert Tsunami Information        Minor      Unlikely
  ptwc       Alert Tsunami Information        Minor      Unlikely


### 3.5 Coordinates, timestamps, and units

### 3.5.1 Coordinate validity (WGS84: latitude −90..90, longitude −180..180)

In [64]:

def check_coords(df, lat, lon, name):
    lat_v = pd.to_numeric(df[lat], errors="coerce")
    lon_v = pd.to_numeric(df[lon], errors="coerce")
    return {
        "source": name,
        "rows": len(df),
        "missing": int((lat_v.isna() | lon_v.isna()).sum()),
        "lat_out_of_range": int(((lat_v < -90) | (lat_v > 90)).sum()),
        "lon_out_of_range": int(((lon_v < -180) | (lon_v > 180)).sum()),
        "at_0_0": int(((lat_v == 0) & (lon_v == 0)).sum()),
    }

pd.DataFrame([
    check_coords(noaa, "latitude", "longitude", "NOAA HazEL"),
    check_coords(usgs_hist, "latitude", "longitude", "USGS FDSN"),
    check_coords(usgs_rt, "lat", "lon", "USGS real-time"),
    check_coords(tsunami_gov, "lat", "lon", "tsunami.gov"),
])


,source,rows,missing,lat_out_of_range,lon_out_of_range,at_0_0
0,NOAA HazEL,221,9,0,0,0
1,USGS FDSN,343,0,0,0,0
2,USGS real-time,726,0,0,0,0
3,tsunami.gov,2,0,0,0,0


### 3.5.2 Timestamp formats and time zones

In [65]:
print("NOAA date components and available precision:")
print(noaa["time_precision"].value_counts().to_string())
print(noaa[["id", "year", "month", "day", "hour", "minute", "second", "time_precision"]].head(3).to_string(index=False))
print("USGS CSV time (ISO 8601 UTC):", usgs_hist["time"].iloc[0])
print("USGS real-time time (Unix milliseconds UTC):", usgs_rt["time"].iloc[0], "->", usgs_rt["time_utc"].iloc[0])
print("tsunami.gov Atom updated (ISO 8601 UTC):", tsunami_gov["updated"].iloc[0])
print("tsunami.gov CAP sent and reported origin:", tsunami_gov["cap_sent"].iloc[0], tsunami_gov["cap_EventOriginTime"].iloc[0])

coverage = pd.DataFrame({
    "first": [noaa["date_utc"].min(), usgs_hist["time_utc"].min(), usgs_rt["time_utc"].min()],
    "last": [noaa["date_utc"].max(), usgs_hist["time_utc"].max(), usgs_rt["time_utc"].max()],
}, index=["NOAA HazEL (date)", "USGS FDSN", "USGS real-time"])
print("Coverage in the saved samples:")
display(coverage)

NOAA date components and available precision:
time_precision
second    189
day        23
minute      9
  id  year  month  day  hour  minute  second time_precision
4522  2010      1    3  21.0    48.0     5.3         second
4523  2010      1    3  22.0    36.0    27.9         second
4542  2010      1    5  12.0    15.0    32.2         second
USGS CSV time (ISO 8601 UTC): 2011-12-26T04:48:08.870Z
USGS real-time time (Unix milliseconds UTC): 1794051338338 -> 2026-11-07 11:35:38.338000+00:00
tsunami.gov Atom updated (ISO 8601 UTC): 2026-11-05T05:01:47Z
tsunami.gov CAP sent and reported origin: 2026-11-05T05:01:47-00:00 2026-11-05T04:56:04-00:00
Coverage in the saved samples:


,first,last
NOAA HazEL (date),2010-01-02 00:00:00.000+00:00,2025-12-12 00:00:00.000+00:00
USGS FDSN,2010-01-02 08:45:33.020+00:00,2011-12-26 04:48:08.870+00:00
USGS real-time,2026-10-08 14:11:53.009+00:00,2026-11-07 11:35:38.338+00:00


### 3.5.3 Units and severity scales

In [66]:
print("Historical USGS magnitude types:")
print(usgs_hist["magType"].value_counts().to_string())
print("Real-time USGS magnitude types:")
print(usgs_rt["magType"].value_counts().to_string())
print("CAP preliminary magnitude types:")
print(tsunami_gov["cap_EventPreliminaryMagnitudeType"].value_counts().to_string())

units = pd.DataFrame([
    ["NOAA HazEL", "eqMagnitude: earthquake magnitude, type absent", "eqDepth: unit not explicit in saved JSON"],
    ["USGS FDSN / real-time", "mag + magType: earthquake magnitude and type", "depth: kilometers"],
    ["tsunami.gov CAP", "EventPreliminaryMagnitude + type: preliminary", "EventDepth: text with its unit"],
], columns=["source", "magnitude", "depth"])
units

Historical USGS magnitude types:
magType
mww    138
mwc    126
mb      42
mwb     34
mw       2
ms       1
Real-time USGS magnitude types:
magType
mb     609
mww    104
mwr      8
mw       2
ml       2
mwb      1
CAP preliminary magnitude types:
cap_EventPreliminaryMagnitudeType
Ml     1
Mwp    1


,source,magnitude,depth
0,NOAA HazEL,"eqMagnitude: earthquake magnitude, type absent",eqDepth: unit not explicit in saved JSON
1,USGS FDSN / real-time,mag + magType: earthquake magnitude and type,depth: kilometers
2,tsunami.gov CAP,EventPreliminaryMagnitude + type: preliminary,EventDepth: text with its unit


## 4. Geographic Scope

The polygon crosses the antimeridian and stores longitudes from about −255 to −66. Only the point-in-polygon test shifts positive event longitudes by 360 degrees. Source coordinates remain in WGS84. Counts below include records with missing coordinates in the before total.

In [67]:
polygon = json.loads(POLYGON.read_text(encoding="utf-8"))
ring = polygon["features"][0]["geometry"]["coordinates"][0]

def in_polygon(lon, lat):
    """Test a point against the polygon, shifting positive longitude across the date line."""
    if pd.isna(lon) or pd.isna(lat):
        return False
    lon, lat = float(lon), float(lat)
    x = lon - 360 if lon > 0 else lon
    inside = False
    for (x1, y1), (x2, y2) in zip(ring, ring[1:]):
        if (y1 > lat) != (y2 > lat) and x < (x2 - x1) * (lat - y1) / (y2 - y1) + x1:
            inside = not inside
    return inside

# Keep WGS84 coordinates in the source tables; transform longitude only inside the test.
for source, frame, lon, lat in [
    ("NOAA HazEL", noaa, "longitude", "latitude"),
    ("USGS FDSN", usgs_hist, "longitude", "latitude"),
    ("USGS real-time", usgs_rt, "lon", "lat"),
    ("tsunami.gov", tsunami_gov, "lon", "lat"),
]:
    frame["in_pacific"] = [in_polygon(x, y) for x, y in zip(frame[lon], frame[lat])]
    missing = int((frame[lon].isna() | frame[lat].isna()).sum())
    outside = len(frame) - int(frame["in_pacific"].sum()) - missing
    print(f"{source}: {len(frame)} total -> {int(frame['in_pacific'].sum())} inside, "
          f"{outside} outside, {missing} without coordinates")

noaa_pacific = noaa.loc[noaa["in_pacific"]].copy()
usgs_hist_pacific = usgs_hist.loc[usgs_hist["in_pacific"]].copy()
usgs_rt_pacific = usgs_rt.loc[usgs_rt["in_pacific"]].copy()
tsunami_gov_pacific = tsunami_gov.loc[tsunami_gov["in_pacific"]].copy()

NOAA HazEL: 221 total -> 205 inside, 7 outside, 9 without coordinates
USGS FDSN: 343 total -> 325 inside, 18 outside, 0 without coordinates
USGS real-time: 726 total -> 544 inside, 182 outside, 0 without coordinates
tsunami.gov: 2 total -> 2 inside, 0 outside, 0 without coordinates


## 5. Incremental Source Comparison

The 7 November morning and afternoon snapshots provide a short interval for revisions. The 12 November snapshot shows a longer interval and message replacement. A missing feed feature is not automatically a deleted earthquake.

In [68]:
from datetime import datetime, timezone

def load_feed(path):
    """Return the feed metadata and its events indexed by id."""
    feed = json.loads(path.read_text(encoding="utf-8"))
    return feed["metadata"], {f["id"]: f for f in feed["features"]}

def utc(ms):
    return datetime.fromtimestamp(ms / 1000, timezone.utc)

first_file = SAMPLES / f"usgs_realtime_{FIRST}.geojson"
second_file = SAMPLES / f"usgs_realtime_{SECOND}.geojson"
meta1, ev1 = load_feed(first_file)
meta2, ev2 = load_feed(second_file)

new_ids = ev2.keys() - ev1.keys()
removed_ids = ev1.keys() - ev2.keys()
modified_ids = sorted(i for i in ev1.keys() & ev2.keys()
                      if ev1[i]["properties"] != ev2[i]["properties"]
                      or ev1[i]["geometry"] != ev2[i]["geometry"])

print("Snapshots:", first_file.name, "->", second_file.name)
print("Records:", len(ev1), "->", len(ev2))
print("New:", len(new_ids), "| Modified:", len(modified_ids), "| Removed:", len(removed_ids))

Snapshots: usgs_realtime_20261107_050434.geojson -> usgs_realtime_20261107_140301.geojson
Records: 714 -> 726
New: 22 | Modified: 25 | Removed: 10


### 5.1 Events absent from the later 7 November feed

The USGS feed is a rolling 30-day window. Compare event origin times with the second snapshot's 30-day cutoff before interpreting an absence.

In [69]:
cutoff_ms = meta2["generated"] - 30 * 24 * 3600 * 1000
expired_ids = [event_id for event_id in removed_ids
               if ev1[event_id]["properties"]["time"] < cutoff_ms]
removed_times = sorted(ev1[event_id]["properties"]["time"] for event_id in removed_ids)

print("Second snapshot generated:", utc(meta2["generated"]), "| 30-day cutoff:", utc(cutoff_ms))
print("Absent events older than the cutoff:", len(expired_ids), "of", len(removed_ids))
print("Origin range of absent events:", utc(removed_times[0]), "->", utc(removed_times[-1]))
print("Oldest event retained:", utc(min(event["properties"]["time"] for event in ev2.values())))

Second snapshot generated: 2026-11-07 14:03:01+00:00 | 30-day cutoff: 2026-10-08 14:03:01+00:00
Absent events older than the cutoff: 10 of 10
Origin range of absent events: 2026-10-08 05:06:50.387000+00:00 -> 2026-10-08 11:36:34.860000+00:00
Oldest event retained: 2026-10-08 14:11:53.009000+00:00


### 5.2 Modified events: what changed?

The comparison includes `geometry` (longitude, latitude, depth), not only `properties`.

In [70]:
field_changes = {}
shifts_km, depth_changes, mag_changes, magtype_changes, only_meta = [], [], [], [], []
for i in modified_ids:
    p1, p2 = ev1[i]["properties"], ev2[i]["properties"]
    changed = {f for f in p1.keys() | p2.keys() if p1.get(f) != p2.get(f)}
    for f in changed:
        field_changes[f] = field_changes.get(f, 0) + 1
    (lon1, lat1, d1), (lon2, lat2, d2) = ev1[i]["geometry"]["coordinates"], ev2[i]["geometry"]["coordinates"]
    if (lon1, lat1, d1) != (lon2, lat2, d2):
        field_changes["geometry"] = field_changes.get("geometry", 0) + 1
        # Equirectangular approximation, accurate enough for shifts of a few kilometers
        dx = (lon2 - lon1) * 111.32 * math.cos(math.radians(lat1))
        dy = (lat2 - lat1) * 110.57
        shifts_km.append(math.hypot(dx, dy))
        depth_changes.append(abs(d2 - d1))
    elif changed <= {"updated", "felt"}:
        only_meta.append(i)
    if p1["mag"] != p2["mag"]:
        mag_changes.append((i, p1["mag"], p2["mag"], p2["magType"]))
    if p1["magType"] != p2["magType"]:
        magtype_changes.append((i, p1["magType"], p2["magType"]))

shifts_km.sort()
print("Events modified:", len(modified_ids))
print("Fields changed (number of events):", dict(sorted(field_changes.items(), key=lambda item: (-item[1], item[0]))))
print(f"Relocated: {len(shifts_km)} | epicenter shift {shifts_km[0]:.1f}-{shifts_km[-1]:.1f} km "
      f"(median {shifts_km[len(shifts_km) // 2]:.1f} km) | largest depth change {max(depth_changes):.1f} km")
print("Magnitude changes (id, before, after, type):", mag_changes)
print("Magnitude-type changes:", magtype_changes or "none")
print("Only 'updated' or 'felt' changed:", len(only_meta))

Events modified: 25
Fields changed (number of events): {'updated': 25, 'dmin': 21, 'geometry': 21, 'nst': 21, 'rms': 21, 'time': 21, 'title': 20, 'place': 18, 'gap': 7, 'mag': 6, 'sig': 6, 'felt': 2, 'mmi': 1, 'types': 1}
Relocated: 21 | epicenter shift 1.5-29.1 km (median 5.9 km) | largest depth change 48.9 km
Magnitude changes (id, before, after, type): [('us6000u35l', 4.5, 4.6, 'mb'), ('us6000u3ch', 4.8, 4.7, 'mb'), ('us6000u4b1', 4.6, 4.5, 'mb'), ('us6000u4m5', 4.9, 5.1, 'mb'), ('us6000u4m6', 4.9, 5, 'mb'), ('us6000u4sv', 4.5, 4.6, 'mb')]
Magnitude-type changes: none
Only 'updated' or 'felt' changed: 4


In [71]:
# How late do revisions arrive? Days between each modified event's origin time and its new `updated`
ages = sorted((ev2[i]["properties"]["updated"] - ev2[i]["properties"]["time"]) / 86_400_000 for i in modified_ids)
print(f"Days from origin to revision: min {ages[0]:.1f}, median {ages[len(ages) // 2]:.1f}, max {ages[-1]:.1f}")

Days from origin to revision: min 1.0, median 18.3, max 28.8


### 5.3 New events

New events are checked against the Pacific polygon of `config/pacific_polygon.geojson` (longitudes stored from −255.5 to −66 because it crosses the antimeridian).

In [72]:
new_mags = sorted(ev2[i]["properties"]["mag"] for i in new_ids)
new_inside = sum(in_polygon(*ev2[i]["geometry"]["coordinates"][:2]) for i in new_ids)
print(f"New events: {len(new_ids)} | magnitude {new_mags[0]}-{new_mags[-1]} | "
      f"M >= 6: {sum(m >= 6 for m in new_mags)} | inside the Pacific polygon: {new_inside}")

New events: 22 | magnitude 4.5-5.5 | M >= 6: 0 | inside the Pacific polygon: 19


### 5.4 The later USGS snapshot (7 → 12 November)

Compare the afternoon 7 November snapshot with 12 November. Count new and revised features, including changes to magnitude, geometry, and `updated`. Classify absences against the later 30-day cutoff; other absences need investigation.

In [73]:
third_file = SAMPLES / f"usgs_realtime_{THIRD}.geojson"
meta3, ev3 = load_feed(third_file)

later_new = ev3.keys() - ev2.keys()
later_absent = ev2.keys() - ev3.keys()
later_modified = sorted(event_id for event_id in ev2.keys() & ev3.keys()
                        if ev2[event_id]["properties"] != ev3[event_id]["properties"]
                        or ev2[event_id]["geometry"] != ev3[event_id]["geometry"])

later_cutoff = meta3["generated"] - 30 * 24 * 3600 * 1000
later_expired = [event_id for event_id in later_absent
                 if ev2[event_id]["properties"]["time"] < later_cutoff]
mag_revisions = [event_id for event_id in later_modified
                 if ev2[event_id]["properties"]["mag"] != ev3[event_id]["properties"]["mag"]]
relocations = [event_id for event_id in later_modified
               if ev2[event_id]["geometry"] != ev3[event_id]["geometry"]]
updated_revisions = [event_id for event_id in later_modified
                     if ev2[event_id]["properties"]["updated"] != ev3[event_id]["properties"]["updated"]]

print("Snapshots:", second_file.name, "->", third_file.name)
print("Records:", len(ev2), "->", len(ev3))
print("New:", len(later_new), "| modified:", len(later_modified), "| absent:", len(later_absent))
print("Absent and past the 30-day cutoff:", len(later_expired))
print("Other absent events (investigate; not proven deleted):", len(later_absent) - len(later_expired))
print("Magnitude revisions:", len(mag_revisions), "| relocations:", len(relocations),
      "| updated changed:", len(updated_revisions))
print("Example magnitude revisions:")
for event_id in mag_revisions[:3]:
    before = ev2[event_id]["properties"]["mag"]
    after = ev3[event_id]["properties"]["mag"]
    print(event_id, before, "->", after)

comparison_counts = pd.DataFrame([
    {"interval": "7 Nov morning → afternoon", "new": len(new_ids),
     "modified": len(modified_ids), "absent": len(removed_ids)},
    {"interval": "7 Nov afternoon → 12 Nov", "new": len(later_new),
     "modified": len(later_modified), "absent": len(later_absent)},
])
comparison_counts

Snapshots: usgs_realtime_20261107_140301.geojson -> usgs_realtime_20261112_033843.geojson
Records: 726 -> 649
New: 121 | modified: 82 | absent: 198
Absent and past the 30-day cutoff: 194
Other absent events (investigate; not proven deleted): 4
Magnitude revisions: 11 | relocations: 68 | updated changed: 82
Example magnitude revisions:
us6000u30x 4.6 -> 4.5
us6000u37h 5.2 -> 5.1
us6000u3c6 5.4 -> 5.5


,interval,new,modified,absent
0,7 Nov morning → afternoon,22,25,10
1,7 Nov afternoon → 12 Nov,121,82,198


**Change comparison.** The bar view compares two different time spans, so its purpose is to show what each saved pair contains, not a per-hour change rate. “Absent” includes events leaving the rolling window.

In [74]:
changes_view = comparison_counts.set_index("interval")
display(changes_view.style.bar(subset=["new", "modified", "absent"], color="#B8754C"))

,new,modified,absent
interval,,,
7 Nov morning → afternoon,22,25,10
7 Nov afternoon → 12 Nov,121,82,198


### 5.5 NTWC and PTWC messages

Compare the Atom entry and its linked CAP message at all three saved snapshots. The feed stores only its current message; an unchanged pair of snapshots cannot rule out a message published and replaced between polls.

In [75]:
bulletin_rows = []
for center in ["ntwc", "ptwc"]:
    for stamp in [FIRST, SECOND, THIRD]:
        atom_file = SAMPLES / f"{center}_{stamp}.xml"
        entries = ET.parse(atom_file).getroot().findall("a:entry", NS)
        for number, entry in enumerate(entries, start=1):
            cap_file = SAMPLES / f"{center}_cap{number}_{stamp}.xml"
            cap = parse_cap(cap_file)
            bulletin_rows.append({
                "center": center,
                "snapshot": stamp,
                "entry_id": entry.findtext("a:id", namespaces=NS),
                "updated": entry.findtext("a:updated", namespaces=NS),
                "cap_id": cap["identifier"],
                "title": entry.findtext("a:title", namespaces=NS),
                "sender": cap["sender"],
                "status": cap["status"],
                "sent": cap["sent"],
                "msg_type": cap["msgType"],
                "event": cap["event"],
                "severity": cap["severity"],
                "urgency": cap["urgency"],
                "certainty": cap["certainty"],
                "headline": cap["headline"],
                "origin": cap["EventOriginTime"],
                "magnitude": cap["EventPreliminaryMagnitude"],
                "depth": cap["EventDepth"],
                "location": cap["EventLatLon"],
            })

bulletin_snapshots = pd.DataFrame(bulletin_rows)
display(bulletin_snapshots[["center", "snapshot", "cap_id", "updated", "severity", "origin", "magnitude"]])

# Compare each center's consecutive snapshots, including CAP content.
cap_fields = ["sender", "status", "sent", "msg_type", "event", "severity",
              "urgency", "certainty", "headline", "origin", "magnitude",
              "depth", "location"]
for center in ["ntwc", "ptwc"]:
    center_rows = bulletin_snapshots.loc[bulletin_snapshots["center"] == center].set_index("snapshot")
    for earlier, later in [(FIRST, SECOND), (SECOND, THIRD)]:
        old = center_rows.loc[earlier]
        new = center_rows.loc[later]
        changed = [field for field in cap_fields if old[field] != new[field]]
        print(f"{center.upper()} {earlier} -> {later}: {old['cap_id']} -> {new['cap_id']}")
        print("  Atom ID changed:", old["entry_id"] != new["entry_id"],
              "| publication time changed:", old["updated"] != new["updated"],
              "| title changed:", old["title"] != new["title"])
        print("  CAP fields changed:", changed if changed else "none")

,center,snapshot,cap_id,updated,severity,origin,magnitude
0,ntwc,20261107_050434,PAAQ-1-to0j1g,2026-11-05T05:01:47Z,Minor,2026-11-05T04:56:04-00:00,4.5
1,ntwc,20261107_140301,PAAQ-1-to0j1g,2026-11-05T05:01:47Z,Minor,2026-11-05T04:56:04-00:00,4.5
2,ntwc,20261112_033843,PAAQ-1-tobe7v,2026-11-11T01:51:32Z,Minor,2026-11-11T01:45:31-00:00,4.7
3,ptwc,20261107_050434,PHEB-1-26297050,2026-10-25T02:27:01Z,Minor,2026-10-24T02:09:00-00:00,7.0
4,ptwc,20261107_140301,PHEB-1-26297050,2026-10-25T02:27:01Z,Minor,2026-10-24T02:09:00-00:00,7.0
5,ptwc,20261112_033843,PHEB-1-26312050,2026-11-08T06:34:57Z,Minor,2026-11-08T06:30:17-00:00,4.6


NTWC 20261107_050434 -> 20261107_140301: PAAQ-1-to0j1g -> PAAQ-1-to0j1g
  Atom ID changed: False | publication time changed: False | title changed: False
  CAP fields changed: none
NTWC 20261107_140301 -> 20261112_033843: PAAQ-1-to0j1g -> PAAQ-1-tobe7v
  Atom ID changed: True | publication time changed: True | title changed: True
  CAP fields changed: ['sent', 'origin', 'magnitude', 'depth', 'location']
PTWC 20261107_050434 -> 20261107_140301: PHEB-1-26297050 -> PHEB-1-26297050
  Atom ID changed: False | publication time changed: False | title changed: False
  CAP fields changed: none
PTWC 20261107_140301 -> 20261112_033843: PHEB-1-26297050 -> PHEB-1-26312050
  Atom ID changed: True | publication time changed: True | title changed: True
  CAP fields changed: ['sent', 'origin', 'magnitude', 'depth', 'location']


## 6. Data Quality Problems

The rows below distinguish missing or inconsistent data from interpretation risks. A missing earthquake magnitude for a landslide-generated tsunami is often not applicable, rather than an error.

In [76]:
quality_rows = []

def add_quality(source, field, problem, count, example_id, consequence):
    quality_rows.append({
        "source": source, "field": field, "problem": problem,
        "count": count, "example_id": example_id, "consequence": consequence,
    })

missing_warning = noaa["warningStatusId"].isna()
add_quality("NOAA HazEL", "warningStatusId", "Historical warning status absent",
            int(missing_warning.sum()), noaa.loc[missing_warning, "id"].iloc[0],
            "Do not infer that no warning was issued.")

questionable = noaa["eventValidity"] <= 2
add_quality("NOAA HazEL", "eventValidity", "Questionable or erroneous tsunami record",
            int(questionable.sum()), noaa.loc[questionable, "id"].iloc[0],
            "Consider validity before using a historical event as evidence.")

partial_time = noaa["time_precision"] != "second"
add_quality("NOAA HazEL", "hour / minute / second", "Time is only partially known",
            int(partial_time.sum()), noaa.loc[partial_time, "id"].iloc[0],
            "A precise timestamp match is not justified.")

missing_coords = noaa["latitude"].isna() | noaa["longitude"].isna()
add_quality("NOAA HazEL", "latitude / longitude", "Coordinates absent",
            int(missing_coords.sum()), noaa.loc[missing_coords, "id"].iloc[0],
            "The polygon cannot classify this record by location.")

empty_mag_error = usgs_hist["magError"].isna()
add_quality("USGS FDSN", "magError", "Magnitude error absent in the CSV sample",
            int(empty_mag_error.sum()), usgs_hist["id"].iloc[0],
            "Uncertainty cannot be assessed from this field.")

flagged = usgs_rt["tsunami"] == 1
add_quality("USGS real-time", "tsunami", "Bulletin indicator can be misread as tsunami occurrence",
            int(flagged.sum()), usgs_rt.loc[flagged, "id"].iloc[0],
            "The flag is not proof that a tsunami occurred.")

add_quality("tsunami.gov", "title / EventDepth", "Free-text location and depth with units in text",
            len(tsunami_gov), tsunami_gov["cap_identifier"].iloc[0],
            "Location and units require explicit parsing.")

ptwc_message = tsunami_gov.loc[tsunami_gov["center"] == "ptwc"].iloc[0]
ptwc_quake = usgs_rt.loc[usgs_rt["id"] == "us6000u4r9"].iloc[0]
origin_gap = ptwc_quake["time_utc"] - ptwc_message["origin_utc"]
add_quality("tsunami.gov", "EventOriginTime", "PTWC CAP origin differs from likely USGS event by one day",
            1, ptwc_message["cap_identifier"],
            f"The candidate USGS event {ptwc_quake['id']} is {origin_gap.total_seconds() / 3600:.2f} hours later.")

quality_problems = pd.DataFrame(quality_rows)
quality_problems

,source,field,problem,count,example_id,consequence
0,NOAA HazEL,warningStatusId,Historical warning status absent,169,4522,Do not infer that no warning was issued.
1,NOAA HazEL,eventValidity,Questionable or erroneous tsunami record,9,5407,Consider validity before using a historical event as evidence.
2,NOAA HazEL,hour / minute / second,Time is only partially known,32,4762,A precise timestamp match is not justified.
3,NOAA HazEL,latitude / longitude,Coordinates absent,9,5739,The polygon cannot classify this record by location.
4,USGS FDSN,magError,Magnitude error absent in the CSV sample,343,usp000jcna,Uncertainty cannot be assessed from this field.
5,USGS real-time,tsunami,Bulletin indicator can be misread as tsunami occurrence,6,us6000u4r9,The flag is not proof that a tsunami occurred.
6,tsunami.gov,title / EventDepth,Free-text location and depth with units in text,2,PAAQ-1-to0j1g,Location and units require explicit parsing.
7,tsunami.gov,EventOriginTime,PTWC CAP origin differs from likely USGS event by one day,1,PHEB-1-26297050,The candidate USGS event us6000u4r9 is 24.01 hours later.


**Missing is not always erroneous.** The following NOAA events have a non-earthquake cause. Their missing `eqMagnitude` or `eqDepth` can be not applicable; do not replace it with an average.

In [77]:
non_earthquake = noaa.loc[~noaa["causeCode"].isin([1, 3, 4, 5])]
print("Non-earthquake causes:", len(non_earthquake))
print("Missing earthquake magnitude in this group:", non_earthquake["eqMagnitude"].isna().sum())
print(non_earthquake["cause"].value_counts().to_string())

Non-earthquake causes: 36
Missing earthquake magnitude in this group: 35
cause
Landslide                17
Meteorological            9
Volcano                   6
Unknown                   2
Volcano and Landslide     2


## 7. Cross-Source Overlap

These are **candidate** links, not confirmed identities. With NOAA seconds, allow ±15 minutes and at most 100 km. With minute or hour precision, use that whole minute or hour as a time interval. With day precision, use that calendar day. Less precise times also require both magnitudes to be present and within 0.5 units, plus the 100 km distance limit. Multiple eligible USGS events are marked as ambiguous.

In [78]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Distance between two WGS84 coordinates, in kilometers."""
    radius = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = p2 - p1
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * radius * math.asin(math.sqrt(a))

def norm_lon(lon):
    """USGS Pacific queries may return longitude above 180."""
    return ((lon + 180) % 360) - 180

def find_pairs(noaa_df, usgs_df):
    candidates = []
    for _, tsunami in noaa_df.dropna(subset=["latitude", "longitude"]).iterrows():
        precision = tsunami["time_precision"]
        if precision == "second":
            start_time = tsunami["time_utc"] - pd.Timedelta(minutes=15)
            end_time = tsunami["time_utc"] + pd.Timedelta(minutes=15)
        else:
            start_time = tsunami["date_utc"]
            if precision in ["hour", "minute"]:
                start_time += pd.Timedelta(hours=int(tsunami["hour"]))
            if precision == "minute":
                start_time += pd.Timedelta(minutes=int(tsunami["minute"]))
            duration = {"minute": "1min", "hour": "1h", "day": "1d"}[precision]
            end_time = start_time + pd.Timedelta(duration)

        if precision == "second":
            nearby = usgs_df.loc[(usgs_df["time_utc"] >= start_time) &
                                 (usgs_df["time_utc"] <= end_time)]
        else:
            nearby = usgs_df.loc[(usgs_df["time_utc"] >= start_time) &
                                 (usgs_df["time_utc"] < end_time)]
        possible = []
        for _, quake in nearby.iterrows():
            distance = haversine_km(tsunami["latitude"], tsunami["longitude"],
                                    quake["latitude"], norm_lon(quake["longitude"]))
            if distance > 100:
                continue
            if precision != "second":
                if pd.isna(tsunami["eqMagnitude"]) or pd.isna(quake["mag"]):
                    continue
                if abs(tsunami["eqMagnitude"] - quake["mag"]) > 0.5:
                    continue
            possible.append((distance, quake))

        if not possible:
            continue
        possible.sort(key=lambda item: item[0])
        distance, quake = possible[0]
        time_difference = None
        if precision == "second":
            time_difference = round((quake["time_utc"] - tsunami["time_utc"]).total_seconds(), 2)
        candidates.append({
            "noaa_id": tsunami["id"], "usgs_id": quake["id"],
            "time_precision": precision, "noaa_date": tsunami["date_utc"].date(),
            "noaa_time": tsunami["time_utc"], "usgs_time": quake["time_utc"],
            "dt_seconds": time_difference, "distance_km": round(distance, 1),
            "noaa_mag": tsunami["eqMagnitude"], "usgs_mag": quake["mag"],
            "usgs_magType": quake["magType"], "noaa_depth": tsunami["eqDepth"],
            "usgs_depth": quake["depth"], "possible_usgs_events": len(possible),
            "place": quake["place"],
        })
    return pd.DataFrame(candidates)

# Use the full requested USGS historical sample period.
start = pd.Timestamp("2010-01-01", tz="UTC")
end = pd.Timestamp("2012-01-01", tz="UTC")
noaa_period = noaa_pacific.loc[
    (noaa_pacific["date_utc"] >= start) &
    (noaa_pacific["date_utc"] < end) &
    noaa_pacific["causeCode"].isin([1, 3, 4, 5])
]
pairs = find_pairs(noaa_period, usgs_hist_pacific)
print("NOAA earthquake-caused tsunamis in 2010–2011:", len(noaa_period))
print("Candidate pairs (NOAA → USGS):", len(pairs))
print("Ambiguous NOAA candidates:", int((pairs["possible_usgs_events"] > 1).sum()))
pairs

NOAA earthquake-caused tsunamis in 2010–2011: 28
Candidate pairs (NOAA → USGS): 28
Ambiguous NOAA candidates: 2


,noaa_id,usgs_id,time_precision,noaa_date,noaa_time,usgs_time,dt_seconds,distance_km,noaa_mag,usgs_mag,usgs_magType,noaa_depth,usgs_depth,possible_usgs_events,place
0,4522,usp000h5nd,second,2010-01-03,2010-01-03 21:48:05.300+00:00,2010-01-03 21:48:02.870+00:00,-2.43,2.2,6.6,6.6,mwc,26.0,10.0,1,"98 km SE of Gizo, Solomon Islands"
1,4523,usp000h5np,second,2010-01-03,2010-01-03 22:36:27.900+00:00,2010-01-03 22:36:25.640+00:00,-2.26,2.0,7.1,7.1,mwc,25.0,10.0,1,"94 km SE of Gizo, Solomon Islands"
2,4542,usp000h5rn,second,2010-01-05,2010-01-05 12:15:32.200+00:00,2010-01-05 12:15:32.210+00:00,0.01,0.0,6.8,6.8,mwc,15.0,15.4,1,"127 km SE of Gizo, Solomon Islands"
3,4682,official20100227063411530_30,second,2010-02-27,2010-02-27 06:34:11.500+00:00,2010-02-27 06:34:11.530+00:00,0.03,0.0,8.8,8.8,mww,22.0,22.9,1,"2010 Maule, Chile Earthquake"
4,4722,usp000h94t,second,2010-03-11,2010-03-11 14:39:43.900+00:00,2010-03-11 14:39:43.950+00:00,0.05,0.0,6.9,6.9,mwc,11.0,11.0,1,"61 km NW of Santa Cruz, Chile"
5,4782,usp000h7qu,second,2010-02-26,2010-02-26 20:31:26.900+00:00,2010-02-26 20:31:26.970+00:00,0.07,0.0,7.0,7.0,mwc,25.0,25.0,1,"70 km SE of Katsuren-haebaru, Japan"
6,5222,usp000hhuc,second,2010-08-10,2010-08-10 05:23:44.900+00:00,2010-08-10 05:23:44.980+00:00,0.08,0.0,7.2,7.3,mwc,25.0,25.0,1,"33 km NW of Port-Vila, Vanuatu"
7,5343,usp000hj2j,second,2010-08-13,2010-08-13 21:19:33.000+00:00,2010-08-13 21:19:34.090+00:00,1.09,0.4,6.9,6.9,mwc,10.0,16.0,1,Mariana Islands region
8,5402,usp000hr97,second,2010-12-21,2010-12-21 17:19:40.600+00:00,2010-12-21 17:19:40.660+00:00,0.06,0.0,7.4,7.4,mwc,14.0,14.0,1,"Bonin Islands, Japan region"
9,5403,usp000hrw0,second,2010-12-25,2010-12-25 13:16:37.000+00:00,2010-12-25 13:16:37.000+00:00,0.00,0.0,7.3,7.3,mwc,16.0,16.0,1,"141 km W of Isangel, Vanuatu"


In [79]:
print("Candidate summary:")
summary = pd.DataFrame({
    "pairs": [len(pairs)],
    "median |dt| (s), exact times": [pairs["dt_seconds"].abs().median()],
    "median distance (km)": [pairs["distance_km"].median()],
    "median |mag difference|": [(pairs["noaa_mag"] - pairs["usgs_mag"]).abs().median()],
})
display(summary)
print("Examples with the largest magnitude differences:")
display(pairs.assign(mag_difference=(pairs["noaa_mag"] - pairs["usgs_mag"]).abs())
        .sort_values("mag_difference", ascending=False)
        .head(5)[["noaa_id", "usgs_id", "time_precision", "dt_seconds",
                  "distance_km", "noaa_mag", "usgs_mag", "usgs_magType",
                  "noaa_depth", "usgs_depth", "possible_usgs_events"]])
print("Candidates needing manual review (more than one eligible USGS event):")
display(pairs.loc[pairs["possible_usgs_events"] > 1,
                  ["noaa_id", "usgs_id", "dt_seconds", "distance_km",
                   "noaa_mag", "usgs_mag", "possible_usgs_events"]])

Candidate summary:


,pairs,"median |dt| (s), exact times",median distance (km),median |mag difference|
0,28,0.075,0.0,0.0


Examples with the largest magnitude differences:


,noaa_id,usgs_id,time_precision,dt_seconds,distance_km,noaa_mag,usgs_mag,usgs_magType,noaa_depth,usgs_depth,possible_usgs_events
24,5931,usp000hvkh,second,0.68,0.0,6.3,6.5,mwc,15.0,15.0,2
10,5406,usp000huc2,second,0.21,15.9,6.8,6.9,mww,28.0,26.0,1
20,5437,usp000j6r4,second,0.31,0.0,7.1,7.2,mww,32.0,32.0,1
21,5438,usp000j7dc,second,0.09,0.0,6.8,6.9,mww,32.0,32.0,1
19,5436,usp000j6rj,second,0.05,0.0,7.0,7.1,mww,28.0,28.0,1


Candidates needing manual review (more than one eligible USGS event):


,noaa_id,usgs_id,dt_seconds,distance_km,noaa_mag,usgs_mag,possible_usgs_events
13,5413,official20110311054624120_30,0.02,0.0,9.1,9.1,3
24,5931,usp000hvkh,0.68,0.0,6.3,6.5,2


### 7.1 tsunami.gov bulletin vs USGS real-time feed

In [80]:

caps = tsunami_gov_pacific.dropna(subset=["cap_EventOriginTime"])
rows = []
for _, t in caps.iterrows():
    t_time = pd.to_datetime(t["cap_EventOriginTime"], utc=True)
    t_lat, t_lon = (float(x) for x in t["cap_EventLatLon"].split()[0].split(","))
    close = usgs_rt_pacific[(usgs_rt_pacific["time_utc"] - t_time).abs() <= pd.Timedelta(minutes=15)]
    for _, u in close.iterrows():
        dist = haversine_km(t_lat, t_lon, u["lat"], u["lon"])
        if dist <= 100:
            rows.append({
                "center": t["center"], "bulletin": t["cap_identifier"], "usgs_id": u["id"],
                "bulletin_time": t_time, "usgs_time": u["time_utc"],
                "dt_seconds": round((u["time_utc"] - t_time).total_seconds(), 1),
                "distance_km": round(dist, 1),
                "bulletin_mag": f"{t['cap_EventPreliminaryMagnitude']} {t['cap_EventPreliminaryMagnitudeType']}",
                "usgs_mag": f"{u['mag']} {u['magType']}",
                "bulletin_depth": t["cap_EventDepth"], "usgs_depth_km": round(u["depth"], 1),
                "usgs_tsunami_flag": u["tsunami"],
                "bulletin_says": t["cap_severity"] + " / " + t["cap_certainty"],
            })
bulletin_pairs = pd.DataFrame(rows)
print("Bulletin/USGS pairs found:", len(bulletin_pairs), "(a message can fall outside the 30-day window of the feed)")
bulletin_pairs.T


Bulletin/USGS pairs found: 1 (a message can fall outside the 30-day window of the feed)


,0
center,ntwc
bulletin,PAAQ-1-to0j1g
usgs_id,nc75462637
bulletin_time,2026-11-05 04:56:04+00:00
usgs_time,2026-11-05 04:56:02.670000+00:00
dt_seconds,-1.3
distance_km,8.7
bulletin_mag,4.5 Ml
usgs_mag,4.96 mw
bulletin_depth,13 kilometers


### 7.2 CAP messages without a strict match

Search within two days and 100 km to diagnose an unmatched bulletin. A result outside the 15-minute rule remains a candidate only.

In [81]:
# Check a two-day window for CAP messages without a strict match.
for _, message in caps.iterrows():
    if message["cap_identifier"] in set(bulletin_pairs.get("bulletin", [])):
        continue
    origin = pd.to_datetime(message["cap_EventOriginTime"], utc=True)
    lat, lon = (float(x) for x in message["cap_EventLatLon"].split()[0].split(","))
    nearby = usgs_rt_pacific[((usgs_rt_pacific["time_utc"] - origin).abs() <= pd.Timedelta(days=2)) & (usgs_rt_pacific["mag"] >= 6)]
    for _, event in nearby.iterrows():
        distance = haversine_km(lat, lon, event["lat"], event["lon"])
        if distance <= 100:
            hours = (event["time_utc"] - origin).total_seconds() / 3600
            print(f"Candidate only: {message['cap_identifier']} / {event['id']} | "
                  f"time difference {hours:+.2f} h | distance {distance:.1f} km | "
                  f"magnitude {message['cap_EventPreliminaryMagnitude']} vs {event['mag']}")

Candidate only: PHEB-1-26297050 / us6000u4r9 | time difference +24.01 h | distance 35.8 km | magnitude 7.0 vs 6.5


## 8. Conclusions

- The saved samples support profiling of all four source families. NOAA and USGS historical samples exceed 200 records; each tsunami.gov feed exposes only its current message.
- NOAA time precision varies by record. Exact timestamps exist only when seconds are known; the original date and any partial clock fields remain available for cautious candidate matching.
- The Pacific polygon classifies location without changing stored WGS84 coordinates. Missing coordinates remain a separate quality issue.
- USGS feed records change after publication; absence from a later rolling feed is not proof of deletion. NTWC/PTWC Atom feeds also replace their current message.
- Cross-source pairs are candidates based on time, distance, and, for partial NOAA times, magnitude. CAP message issue time, reported earthquake origin, and source update time must remain distinct. Magnitude types and depth units need to be preserved.